Rúbrica: 

A continuación se muestra la rúbrica con la que se va a corregir el examen:


| Apartado/Criterio | Ponderación | 
| :-- | --- | 
| Ej. 1.1. Ha tratado de manera adecuada los datos de las columnas. | 1 | 
| Ej. 1.1. Ha seguido un criterio adecuado para elegir las entradas del problema. | 1 |
| Ej. 1.2. Ha diseñado bien la red y el sistema de entrenamiento | 1 | 
| Ej. 1.2. Ha dimensionado bien la red neuronal. | 1 | 
| Ej. 1.2. Ha hecho modificaciones coherentes para conseguir un mejor resultado. | 0,5 | 
| Ej. 1.2. Ha usado su experiencia para valorar si el resultado es válido o no. | 1 | 
| Ej. 2.1. Ha cargado adecuadamente los datos. | 0,5 | 
| Ej. 2.2. Ha diseñado bien la red y el sistema de entrenamiento | 1 | 
| Ej. 2.2. Ha dimensionado bien la red neuronal. | 1 | 
| Ej. 2.2. Ha hecho modificaciones coherentes para conseguir un mejor resultado. | 1 | 
| Ej. 2.2. Ha usado su experiencia para valorar si el resultado es válido o no. | 1 | 



In [1]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow import keras
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.metrics import classification_report
import datetime
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

## Ejercicio 1

In [2]:
vuelos = pd.read_csv("vuelos_pakistan.csv")
vuelos.head()

,Flight_ID,Date,Month,Day_of_Week,Departure_City,Arrival_City,Route_Type,Aircraft_Type,Flight_Duration_Minutes,Passengers,...,Load_Factor_%,Ticket_Price_USD,Delay_Minutes,Delay_Category,On_Time_Status,Weather_Condition,Fuel_Consumption,CO2_Emissions,Customer_Rating,Customer_Feedback
0,PK2026_0001,2026-06-09,June,Tuesday,Jeddah,Islamabad,International,Airbus A320,83.0,120,...,66.67,1140.0,220,Severe,Delayed,Clear,6265l,15662.5kg,4.1,Dreadful customer support
1,PK2026_0002,2026-08-12,August,Wednesday,Dubai,Kuala Lumpur,International,Airbus A320,284.0,179,...,99.44,773.0,27,Minor,Delayed,NaN,3516l,8790.0kg,3.6,"Standard flight, nothing special"
2,PK2026_0003,2026-04-20,April,Monday,Doha,Lahore,International,ATR 72,333.0,69,...,98.57,155.0,176,Severe,Delayed,Fog,13538l,33845.0kg,3.0,Tardy arrival but very cozy
3,PK2026_0004,2026-12-07,December,Monday,Jeddah,Lahore,International,Boeing 777,330.0,291,...,83.14,1237.0,87,Moderate,Delayed,NaN,18850l,47125.0kg,NaN,Mediocre experience overall
4,PK2026_0005,2026-05-04,May,Monday,Lahore,Doha,International,Boeing 737,283.0,159,...,99.38,141.0,82,Moderate,Delayed,NaN,13474l,33685.0kg,3.0,Behind schedule but quite relaxed


In [3]:
vuelos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Flight_ID                800 non-null    object 
 1   Date                     800 non-null    object 
 2   Month                    800 non-null    object 
 3   Day_of_Week              800 non-null    object 
 4   Departure_City           800 non-null    object 
 5   Arrival_City             800 non-null    object 
 6   Route_Type               800 non-null    object 
 7   Aircraft_Type            800 non-null    object 
 8   Flight_Duration_Minutes  725 non-null    float64
 9   Passengers               800 non-null    int64  
 10  Seat_Capacity            800 non-null    int64  
 11  Load_Factor_%            800 non-null    float64
 12  Ticket_Price_USD         786 non-null    float64
 13  Delay_Minutes            800 non-null    int64  
 14  Delay_Category           8

In [4]:
vuelos.isnull().sum()

Flight_ID                    0
Date                         0
Month                        0
Day_of_Week                  0
Departure_City               0
Arrival_City                 0
Route_Type                   0
Aircraft_Type                0
Flight_Duration_Minutes     75
Passengers                   0
Seat_Capacity                0
Load_Factor_%                0
Ticket_Price_USD            14
Delay_Minutes                0
Delay_Category               0
On_Time_Status               0
Weather_Condition          279
Fuel_Consumption             0
CO2_Emissions                0
Customer_Rating            127
Customer_Feedback            0
dtype: int64

In [5]:
vuelos['Customer_Rating'] = vuelos['Customer_Rating'].fillna(vuelos['Customer_Rating'].mean())

In [6]:
vuelos.isnull().sum()

Flight_ID                    0
Date                         0
Month                        0
Day_of_Week                  0
Departure_City               0
Arrival_City                 0
Route_Type                   0
Aircraft_Type                0
Flight_Duration_Minutes     75
Passengers                   0
Seat_Capacity                0
Load_Factor_%                0
Ticket_Price_USD            14
Delay_Minutes                0
Delay_Category               0
On_Time_Status               0
Weather_Condition          279
Fuel_Consumption             0
CO2_Emissions                0
Customer_Rating              0
Customer_Feedback            0
dtype: int64

In [7]:
vuelos = vuelos.drop(columns=["Flight_ID", "Date", "Weather_Condition", "Customer_Feedback"])

In [8]:
vuelos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Month                    800 non-null    object 
 1   Day_of_Week              800 non-null    object 
 2   Departure_City           800 non-null    object 
 3   Arrival_City             800 non-null    object 
 4   Route_Type               800 non-null    object 
 5   Aircraft_Type            800 non-null    object 
 6   Flight_Duration_Minutes  725 non-null    float64
 7   Passengers               800 non-null    int64  
 8   Seat_Capacity            800 non-null    int64  
 9   Load_Factor_%            800 non-null    float64
 10  Ticket_Price_USD         786 non-null    float64
 11  Delay_Minutes            800 non-null    int64  
 12  Delay_Category           800 non-null    object 
 13  On_Time_Status           800 non-null    object 
 14  Fuel_Consumption         8

In [9]:
vuelos.head()

,Month,Day_of_Week,Departure_City,Arrival_City,Route_Type,Aircraft_Type,Flight_Duration_Minutes,Passengers,Seat_Capacity,Load_Factor_%,Ticket_Price_USD,Delay_Minutes,Delay_Category,On_Time_Status,Fuel_Consumption,CO2_Emissions,Customer_Rating
0,June,Tuesday,Jeddah,Islamabad,International,Airbus A320,83.0,120,180,66.67,1140.0,220,Severe,Delayed,6265l,15662.5kg,4.100000
1,August,Wednesday,Dubai,Kuala Lumpur,International,Airbus A320,284.0,179,180,99.44,773.0,27,Minor,Delayed,3516l,8790.0kg,3.600000
2,April,Monday,Doha,Lahore,International,ATR 72,333.0,69,70,98.57,155.0,176,Severe,Delayed,13538l,33845.0kg,3.000000
3,December,Monday,Jeddah,Lahore,International,Boeing 777,330.0,291,350,83.14,1237.0,87,Moderate,Delayed,18850l,47125.0kg,3.736107
4,May,Monday,Lahore,Doha,International,Boeing 737,283.0,159,160,99.38,141.0,82,Moderate,Delayed,13474l,33685.0kg,3.000000


In [10]:
vuelos["Fuel_Consumption"] = vuelos["Fuel_Consumption"].str.replace('l', '').astype(int)
vuelos["CO2_Emissions"] = vuelos["CO2_Emissions"].str.replace('kg', '').astype(float)

In [11]:
vuelos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Month                    800 non-null    object 
 1   Day_of_Week              800 non-null    object 
 2   Departure_City           800 non-null    object 
 3   Arrival_City             800 non-null    object 
 4   Route_Type               800 non-null    object 
 5   Aircraft_Type            800 non-null    object 
 6   Flight_Duration_Minutes  725 non-null    float64
 7   Passengers               800 non-null    int64  
 8   Seat_Capacity            800 non-null    int64  
 9   Load_Factor_%            800 non-null    float64
 10  Ticket_Price_USD         786 non-null    float64
 11  Delay_Minutes            800 non-null    int64  
 12  Delay_Category           800 non-null    object 
 13  On_Time_Status           800 non-null    object 
 14  Fuel_Consumption         8

In [12]:
# Filtra por tipos 'object' (texto) o 'category'
columnas_cat = vuelos.select_dtypes(include=['object', 'category']).columns

for col in columnas_cat:
    print(f"Valores únicos en {col}:")
    print(vuelos[col].unique())
    print("-" * 20)


Valores únicos en Month:
['June' 'August' 'April' 'December' 'May' 'January' 'September' 'November'
 'July' 'March' 'February' 'October']
--------------------
Valores únicos en Day_of_Week:
['Tuesday' 'Wednesday' 'Monday' 'Saturday' 'Friday' 'Sunday' 'Thursday']
--------------------
Valores únicos en Departure_City:
['Jeddah' 'Dubai' 'Doha' 'Lahore' 'Islamabad' 'Karachi' 'Kuala Lumpur'
 'London']
--------------------
Valores únicos en Arrival_City:
['Islamabad' 'Kuala Lumpur' 'Lahore' 'Doha' 'Dubai' 'London' 'Karachi'
 'Jeddah']
--------------------
Valores únicos en Route_Type:
['International' 'Domestic']
--------------------
Valores únicos en Aircraft_Type:
['Airbus A320' 'ATR 72' 'Boeing 777' 'Boeing 737']
--------------------
Valores únicos en Delay_Category:
['Severe' 'Minor' 'Moderate' 'No Delay']
--------------------
Valores únicos en On_Time_Status:
['Delayed' 'On Time']
--------------------


In [13]:
vuelos = pd.get_dummies(vuelos,dtype=int)

In [14]:
vuelos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 56 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Flight_Duration_Minutes      725 non-null    float64
 1   Passengers                   800 non-null    int64  
 2   Seat_Capacity                800 non-null    int64  
 3   Load_Factor_%                800 non-null    float64
 4   Ticket_Price_USD             786 non-null    float64
 5   Delay_Minutes                800 non-null    int64  
 6   Fuel_Consumption             800 non-null    int64  
 7   CO2_Emissions                800 non-null    float64
 8   Customer_Rating              800 non-null    float64
 9   Month_April                  800 non-null    int64  
 10  Month_August                 800 non-null    int64  
 11  Month_December               800 non-null    int64  
 12  Month_February               800 non-null    int64  
 13  Month_January       

In [15]:
vuelos.head()

,Flight_Duration_Minutes,Passengers,Seat_Capacity,Load_Factor_%,Ticket_Price_USD,Delay_Minutes,Fuel_Consumption,CO2_Emissions,Customer_Rating,Month_April,...,Aircraft_Type_ATR 72,Aircraft_Type_Airbus A320,Aircraft_Type_Boeing 737,Aircraft_Type_Boeing 777,Delay_Category_Minor,Delay_Category_Moderate,Delay_Category_No Delay,Delay_Category_Severe,On_Time_Status_Delayed,On_Time_Status_On Time
0,83.0,120,180,66.67,1140.0,220,6265,15662.5,4.100000,0,...,0,1,0,0,0,0,0,1,1,0
1,284.0,179,180,99.44,773.0,27,3516,8790.0,3.600000,0,...,0,1,0,0,1,0,0,0,1,0
2,333.0,69,70,98.57,155.0,176,13538,33845.0,3.000000,1,...,1,0,0,0,0,0,0,1,1,0
3,330.0,291,350,83.14,1237.0,87,18850,47125.0,3.736107,0,...,0,0,0,1,0,1,0,0,1,0
4,283.0,159,160,99.38,141.0,82,13474,33685.0,3.000000,0,...,0,0,1,0,0,1,0,0,1,0


In [24]:
vuelos = vuelos.dropna()

In [25]:
vuelos.isnull().sum()

Flight_Duration_Minutes        0
Passengers                     0
Seat_Capacity                  0
Load_Factor_%                  0
Ticket_Price_USD               0
Delay_Minutes                  0
Fuel_Consumption               0
CO2_Emissions                  0
Customer_Rating                0
Month_April                    0
Month_August                   0
Month_December                 0
Month_February                 0
Month_January                  0
Month_July                     0
Month_June                     0
Month_March                    0
Month_May                      0
Month_November                 0
Month_October                  0
Month_September                0
Day_of_Week_Friday             0
Day_of_Week_Monday             0
Day_of_Week_Saturday           0
Day_of_Week_Sunday             0
Day_of_Week_Thursday           0
Day_of_Week_Tuesday            0
Day_of_Week_Wednesday          0
Departure_City_Doha            0
Departure_City_Dubai           0
Departure_

In [26]:
vuelos.corr()["Ticket_Price_USD"].abs().sort_values(ascending=False)[1:]

Day_of_Week_Thursday           0.096530
Arrival_City_Jeddah            0.080967
Month_June                     0.076779
Departure_City_Karachi         0.075604
Passengers                     0.058469
Day_of_Week_Tuesday            0.056840
Month_September                0.056730
Departure_City_Lahore          0.055408
Month_April                    0.053136
Month_March                    0.049383
Departure_City_Dubai           0.042675
Month_August                   0.040593
Arrival_City_Dubai             0.038692
Seat_Capacity                  0.036270
Arrival_City_Karachi           0.034977
Route_Type_International       0.034601
Route_Type_Domestic            0.034601
Aircraft_Type_Boeing 777       0.031576
Delay_Category_Minor           0.030556
Departure_City_London          0.030416
Month_December                 0.029213
Departure_City_Jeddah          0.028708
Delay_Category_No Delay        0.028627
Delay_Category_Moderate        0.028421
Month_January                  0.026789


In [27]:
X = vuelos.drop(["Ticket_Price_USD"],axis=1)
y = vuelos["Ticket_Price_USD"]

In [28]:
escalador = StandardScaler()
X = escalador.fit_transform(X)
y = escalador.fit_transform(y.to_frame())

In [29]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X,y,test_size=0.1,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full,y_train_full,test_size=0.1,random_state=42)

In [30]:
X_train.shape[1:]

(55,)

Pruebo con un RandomForest Regresion

In [31]:
from sklearn.ensemble import RandomForestRegressor
rnd_reg = RandomForestRegressor(n_estimators=1000, n_jobs=-1,random_state=42)
rnd_reg.fit(X_train,y_train)
y_pred = rnd_reg.predict(X_test)
from sklearn.metrics import r2_score
r2_score(y_test,y_pred)

c:\Users\Carmen\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


-0.0632369679092748

Se crea una red neuronal normal en regresión

In [32]:
model = keras.models.Sequential()
model.add(keras.layers.Dense(25,input_shape=X_train.shape[1:],activation="relu"))
model.add(keras.layers.Dense(5,activation="relu"))
model.add(keras.layers.Dense(1))


c:\Users\Carmen\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [68]:
# Red Neuronal normal pero con BatchNormalization y inicializador

model = keras.models.Sequential()

# 1. Entrada: Asegúrate de que X_train venga de las columnas del CSV
model.add(keras.layers.InputLayer(shape=X_train.shape[1:]))
# 2. Bloques ocultos (Igual que antes, esto es el "cerebro")
model.add(keras.layers.Dense(25, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(5, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

# 3. Capa de SALIDA para REGRESIÓN: 
# IMPORTANTE: 1 sola neurona (el precio) y SIN activación (o activación lineal)
model.add(keras.layers.Dense(1)) 

# 4. Compilación para REGRESIÓN:
# Cambiamos loss a 'mse' (error cuadrático medio) o 'mae'
model.compile(loss='mean_squared_error', optimizer=keras.optimizers.Nadam(learning_rate=0.001), metrics=['mae'])

# 5. Entrenamiento
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

history = model.fit(X_train, y_train, epochs=1000, validation_split=0.2,callbacks=[early_stopping_cb],batch_size=32)

Epoch 1/1000
172/172 ━━━━━━━━━━━━━━━━━━━━ 12s 64ms/step - loss: 883.2402 - mae: 25.4866 - val_loss: 901.0627 - val_mae: 25.7937
Epoch 2/1000
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - loss: 831.2986 - mae: 24.4853 - val_loss: 843.9935 - val_mae: 24.7133
Epoch 3/1000
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - loss: 750.5200 - mae: 22.9334 - val_loss: 785.3641 - val_mae: 23.6056
Epoch 4/1000
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 62ms/step - loss: 651.9783 - mae: 21.0563 - val_loss: 684.7315 - val_mae: 21.7153
Epoch 5/1000
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - loss: 548.6688 - mae: 19.0851 - val_loss: 556.4750 - val_mae: 19.3283
Epoch 6/1000
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 62ms/step - loss: 452.9737 - mae: 17.2864 - val_loss: 419.5580 - val_mae: 16.7332
Epoch 7/1000
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - loss: 373.1676 - mae: 15.7957 - val_loss: 350.9808 - val_mae: 15.4024
Epoch 8/1000
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - loss: 314.1922 - mae: 14.7143 - val_loss: 303.

In [35]:
model.compile(loss="mean_squared_error", optimizer=keras.optimizers.SGD(learning_rate=0.0005), metrics=["mse"])

In [36]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)

In [37]:
history = model.fit(X_train,y_train, epochs=1000,validation_data=(X_val,y_val),callbacks=[early_stopping_cb])

Epoch 1/1000
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.4418 - mse: 1.4418 - val_loss: 1.9299 - val_mse: 1.9299
Epoch 2/1000
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.3709 - mse: 1.3709 - val_loss: 1.8414 - val_mse: 1.8414
Epoch 3/1000
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.3142 - mse: 1.3142 - val_loss: 1.7660 - val_mse: 1.7660
Epoch 4/1000
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.2681 - mse: 1.2681 - val_loss: 1.7015 - val_mse: 1.7015
Epoch 5/1000
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.2293 - mse: 1.2293 - val_loss: 1.6450 - val_mse: 1.6450
Epoch 6/1000
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.1972 - mse: 1.1972 - val_loss: 1.5969 - val_mse: 1.5969
Epoch 7/1000
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.1700 - mse: 1.1700 - val_loss: 1.5549 - val_mse: 1.5549
Epoch 8/1000
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.1472 - mse: 1.1472 - val_loss: 1.5187 - val_mse: 1.5187
Epoch 9/1000
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - lo

In [69]:
model.evaluate(X_test,y_test)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 234.4357 - mae: 13.2647


[234.4357452392578, 13.264700889587402]

## Ejercicio 2

In [70]:
folders = listdir('./cartas')

photos = []
labels = []

for idx,folder in enumerate(folders): # Si queremos coger un numero de imagenes en concreto
    for file in listdir('./cartas/'+folder):
        photo = load_img('./cartas/'+folder+'/' +file, color_mode='grayscale', target_size=(117, 117))
        photo = img_to_array(photo)
        photos.append(photo)
        labels.append(float(idx))
        del photo
    print (idx)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52


In [71]:
photos_array = asarray(photos)
labels_array = asarray(labels)

In [72]:
X = np.asarray(photos)
y = np.asarray(labels)

In [73]:
X.shape

(7624, 117, 117, 1)

In [74]:
X = X.reshape(7624,-1) # Si hago flatten, esto no se ejecuta.

In [75]:
X = X/255.0

In [76]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X,y,test_size=0.1,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full,y_train_full,test_size=0.1,random_state=42)

In [77]:
X_train.shape[1:]

(13689,)

Pruebo con un random forest classifier antes de meter red neuronal

In [78]:
from sklearn.ensemble import RandomForestClassifier
rnd_clf = RandomForestClassifier(n_estimators=1000, n_jobs=-1,random_state=42,bootstrap=False)
rnd_clf.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1000
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",False
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metr

In [79]:
y_pred = rnd_clf.predict(X_test)

In [80]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.4927916120576671

Pruebo con una red neuronal normal

In [82]:
model = keras.models.Sequential()
#model.add(keras.layers.Flatten(input_shape=X_train.shape[1:])) Si no hubiese hecho el reshape
model.add(keras.layers.Dense(3000,input_shape=X_train.shape[1:], activation="relu"))
model.add(keras.layers.Dense(750,activation="relu"))
model.add(keras.layers.Dense(150,activation="relu"))
model.add(keras.layers.Dense(53,activation="softmax"))

In [87]:
model = keras.models.Sequential()

# 1. Capa de entrada
model.add(keras.layers.InputLayer(shape=X_train.shape[1:]))

# 2. Bloques de capas densas con BN e Inicializadores
# Usamos las neuronas que te pide tu esquema (300, 200, 100, 20)
model.add(keras.layers.Dense(300, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(200, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(100, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(20, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

# 3. Capa de salida (CORREGIDA)
# - 53 neuronas (una por carta)
# - 'softmax' para que nos dé probabilidades
# - 'glorot_normal' es el estándar para softmax
model.add(keras.layers.Dense(53, activation='softmax', kernel_initializer='glorot_normal'))

# 4. Compilación (CORREGIDA)
# - 'sparse_categorical_crossentropy' porque tus y_train son números enteros
model.compile(loss='sparse_categorical_crossentropy', optimizer=keras.optimizers.Nadam(learning_rate=0.001), metrics=['accuracy'])

# 5. Entrenamiento
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

history = model.fit(X_train, y_train, epochs=100, validation_data=(X_val, y_val), callbacks=[early_stopping_cb],batch_size=256)

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0860 - loss: 3.7453 - val_accuracy: 0.0291 - val_loss: 11.5065
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2156 - loss: 3.2467 - val_accuracy: 0.0291 - val_loss: 5.0162
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3039 - loss: 2.9410 - val_accuracy: 0.0553 - val_loss: 4.0974
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3845 - loss: 2.6822 - val_accuracy: 0.0728 - val_loss: 4.0072
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4580 - loss: 2.4571 - val_accuracy: 0.1237 - val_loss: 3.5270
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5363 - loss: 2.2108 - val_accuracy: 0.0728 - val_loss: 3.8978
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6093 - loss: 1.9628 - val_accuracy: 0.1252 - val_loss: 3.6479
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6796 - loss: 1.7345 - val_accuracy: 0

In [83]:
model.compile(loss="sparse_categorical_crossentropy", optimizer=keras.optimizers.SGD(learning_rate=0.0005), metrics=["accuracy"])

In [84]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)

In [85]:
history = model.fit(X_train,y_train, epochs=5,validation_data=(X_val,y_val),callbacks=[early_stopping_cb])

Epoch 1/5
193/193 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.0442 - loss: 3.9513 - val_accuracy: 0.0437 - val_loss: 3.8843
Epoch 2/5
193/193 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.0768 - loss: 3.8308 - val_accuracy: 0.0626 - val_loss: 3.8153
Epoch 3/5
193/193 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.0894 - loss: 3.7555 - val_accuracy: 0.0786 - val_loss: 3.7415
Epoch 4/5
193/193 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.1116 - loss: 3.6876 - val_accuracy: 0.1179 - val_loss: 3.6764
Epoch 5/5
193/193 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.1200 - loss: 3.6259 - val_accuracy: 0.1412 - val_loss: 3.6115


In [88]:
model.evaluate(X_test,y_test)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1088 - loss: 3.6009 


[3.600891590118408, 0.1087811291217804]

Probamos con una red neuronal convolucional

In [89]:
X = photos_array/255.0
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [90]:
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten
model = keras.models.Sequential()
model.add(Conv2D(16,(3,3), activation='relu', input_shape = (117,117,1)))
model.add(MaxPool2D(2,2))
model.add(Conv2D(32,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Conv2D(64,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Flatten())
model.add(keras.layers.Dense(100,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(30,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(53,activation='softmax',kernel_initializer='glorot_normal'))

c:\Users\Carmen\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [91]:
#Vamos a usar Adam como optimizador y entrenamos.
model.compile(loss='crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.01, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=15,validation_split = 0.1,callbacks=[early_stopping_cb],batch_size=256)

Epoch 1/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step - accuracy: 0.0194 - loss: 4.3061 - val_accuracy: 0.0291 - val_loss: 3.9685
Epoch 2/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 93ms/step - accuracy: 0.0198 - loss: 3.9667 - val_accuracy: 0.0146 - val_loss: 3.9681
Epoch 3/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 92ms/step - accuracy: 0.0215 - loss: 3.9651 - val_accuracy: 0.0146 - val_loss: 3.9687
Epoch 4/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 93ms/step - accuracy: 0.0235 - loss: 3.9646 - val_accuracy: 0.0146 - val_loss: 3.9687
Epoch 5/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 92ms/step - accuracy: 0.0235 - loss: 3.9644 - val_accuracy: 0.0146 - val_loss: 3.9698
Epoch 6/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 92ms/step - accuracy: 0.0235 - loss: 3.9642 - val_accuracy: 0.0146 - val_loss: 3.9692
Epoch 7/15
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 91ms/step - accuracy: 0.0212 - loss: 3.9643 - val_accuracy: 0.0146 - val_loss: 3.9689


In [92]:
model.evaluate(X_test,y_test)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0210 - loss: 3.9659


[3.965874671936035, 0.020969856530427933]